# Multiple Variable Linear Regression

In this lab, you will extend the data structures and previously developed routines to support multiple features. Several routines are updated making the lab appear lengthy, but it makes minor adjustments to previous routines making it quick to review.

In [18]:
import numpy as np
import copy, math 
import matplotlib.pyplot as plt 
plt.style.use('/Users/cheerupdimbo/Documents/ds-sprint-july/ds-sprint-july/week3_ml_core/deeplearning.mplstyle')

## Role of Alpha (Learning Rate) in Gradient Descent

Alpha ($\alpha$) controls the **step size** in each gradient descent update:

$$w_j \leftarrow w_j - \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \qquad b \leftarrow b - \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}$$

The gradient $\frac{\partial J}{\partial w_j}$ tells you the **direction** to move (specifically, the direction of steepest *increase* in cost — which is why you subtract it, to move toward the minimum). Alpha tells you **how far** to move in that direction on each step.

### Why the size of alpha matters

- **Too small**: gradient descent still converges to the minimum, but takes a very large number of iterations to get there — slow, but safe.
- **Too large**: each step overshoots the minimum. Instead of converging, cost can oscillate or diverge (increase every iteration instead of decreasing) — the algorithm fails to find the minimum at all.
- **Well-tuned**: cost decreases smoothly and reasonably quickly toward the minimum, iteration after iteration.

### How alpha is typically chosen in practice

There's no formula that gives you the "correct" alpha in advance — it depends on the shape of $J(\mathbf{w},b)$, which depends on your specific data and features. The standard approach:

1. Try a range of values, often scaled by ~3x each step: `0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1`
2. For each, run a small number of iterations and plot cost vs. iteration number
3. Pick the largest alpha where cost decreases smoothly and quickly, without oscillating or increasing

### Connection to feature scaling

Alpha interacts with the *scale* of your features. If features have very different ranges (e.g. one feature 0-1, another 0-1000), the cost function becomes elongated/skewed, and a single alpha that works well for one dimension may be too large or too small for another — which is why feature scaling (normalization/standardization) is typically done before gradient descent, so a single alpha can work reasonably well across all features.

<a name="toc_15456_1.3"></a>
## 1.3 Notation
Here is a summary of some of the notation you will encounter, updated for multiple features.  

| General <br /> Notation | Description | Python (if applicable) |
|:---|:---|:---|
| $a$ | scalar, non bold | |
| $\mathbf{a}$ | vector, bold | |
| $\mathbf{A}$ | matrix, bold capital | |
| **Regression** | | |
| $\mathbf{X}$ | training example matrix | `X_train` |
| $\mathbf{y}$ | training example targets | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | $i_{th}$ Training Example | `X[i]`, `y[i]` |
| m | number of training examples | `m` |
| n | number of features in each example | `n` |
| $\mathbf{w}$ | parameter: weight | `w` |
| $b$ | parameter: bias | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | The result of the model evaluation at $\mathbf{x^{(i)}}$ parameterized by $\mathbf{w},b$: $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |


In [19]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

# data is stored in numpy array/matrix
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

X Shape: (3, 4), X Type:<class 'numpy.ndarray'>)
[[2104    5    1   45]
 [1416    3    2   40]
 [ 852    2    1   35]]
y Shape: (3,), y Type:<class 'numpy.ndarray'>)
[460 232 178]


In [20]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

w_init shape: (4,), b_init type: <class 'float'>


<a name="toc_15456_3"></a>
# 3 Model Prediction With Multiple Variables
The model's prediction with multiple variables is given by the linear model:

*(1)*:
$$f_{\mathbf{w},b}(\mathbf{x}) = w_0x_0 + w_1x_1 + ... + w_{n-1}x_{n-1} + b$$

or in vector notation:

*(2)*:
$$f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b$$

where $\cdot$ is a vector `dot product`

To demonstrate the dot product, we will implement prediction using (1) and (2).

In [21]:
# for equation 1
def predict_sum(x, w, b):
    n = x.shape[0]
    p = 0
    for i in range(n):
      p_i = x[i] * w[i]
      p += p_i
    p += b

# for equation 2
def predict_vector(x, w, b): 
    p = np.dot(x, w) + b     
    return p    

<a name="toc_15456_4"></a>
# 4 Compute Cost With Multiple Variables
The equation for the cost function with multiple variables $J(\mathbf{w},b)$ is:

*(3)*:
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2$$

where:

*(4)*:
$$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b$$

In contrast to previous labs, $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are vectors rather than scalars supporting multiple features.

In [22]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        # equation 4
        f_wb_i = np.dot(X[i], w) + b

        # equation 3
        cost +=  (f_wb_i - y[i])**2
    cost /= 2*m 
    return cost

cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w: {cost}')

Cost at optimal w: 1.5578904880036537e-12


<a name="toc_15456_5"></a>
# 5 Gradient Descent With Multiple Variables
Gradient descent for multiple variables:

*(5)*:
$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where, n is the number of features, parameters $w_j$,  $b$, are updated simultaneously and where  

*(6)*:
$$
\begin{align*}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)}  \\
\end{align*}
$$
*(7)*:
$$
\begin{align*}
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})
\end{align*}
$$

* m is the number of training examples in the data set
    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$ is the target value

<a name="toc_15456_5.1"></a>
## 5.1 Compute Gradient with Multiple Variables
An implementation for calculating the equations (6) and (7) is below. There are many ways to implement this. In this version, there is an
- outer loop over all m examples. 
    - $\frac{\partial J(\mathbf{w},b)}{\partial b}$ for the example can be computed directly and accumulated
    - in a second loop over all n features:
        - $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ is computed for each $w_j$.

## Vectorized Gradient Computation — Worked Example

Concrete example with `m=3` examples, `n=2` features.

### Setup

$$X = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}_{(3\times2)} \quad \mathbf{w} = \begin{bmatrix} 0.1 \\ 0.2 \end{bmatrix}_{(2,)} \quad b = 0.5 \quad \mathbf{y} = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}_{(3,)}$$

### Step 1 — Error vector: `err = X @ w + b - y`

$X\mathbf{w}$ (matrix-vector product, each row dotted with $\mathbf{w}$):

$$X\mathbf{w} = \begin{bmatrix} 1(0.1)+2(0.2) \\ 3(0.1)+4(0.2) \\ 5(0.1)+6(0.2) \end{bmatrix} = \begin{bmatrix} 0.5 \\ 1.1 \\ 1.7 \end{bmatrix}$$

Add $b$, subtract $\mathbf{y}$:

$$\mathbf{err} = \begin{bmatrix} 0.5+0.5-1 \\ 1.1+0.5-2 \\ 1.7+0.5-3 \end{bmatrix} = \begin{bmatrix} 0.0 \\ -0.4 \\ -0.8 \end{bmatrix}_{(3,)}$$

This one array holds all 3 per-example errors — what the outer loop builds up one at a time.

### Step 2 — Gradient w.r.t. w: `dj_dw = (X.T @ err) / m`

$X^T$ flips $X$ to $(2\times3)$:

$$X^T = \begin{bmatrix} 1 & 3 & 5 \\ 2 & 4 & 6 \end{bmatrix}$$

$X^T\mathbf{err}$ — each row of $X^T$ dotted with $\mathbf{err}$:

$$X^T\mathbf{err} = \begin{bmatrix} 1(0.0)+3(-0.4)+5(-0.8) \\ 2(0.0)+4(-0.4)+6(-0.8) \end{bmatrix} = \begin{bmatrix} -5.2 \\ -6.4 \end{bmatrix}$$

Divide by $m=3$:

$$\frac{\partial J}{\partial \mathbf{w}} = \begin{bmatrix} -1.733 \\ -2.133 \end{bmatrix}_{(2,)}$$

The top entry, $-5.2$, is exactly $1(0.0) + 3(-0.4) + 5(-0.8)$ — that's `err[0]*X[0,0] + err[1]*X[1,0] + err[2]*X[2,0]`, which is precisely what the **inner loop** accumulates into `dj_dw[0]` across all 3 iterations of `i`, for `j=0`. The matrix multiply performs that same accumulation for both columns ($j=0$ and $j=1$) in one shot.

### Step 3 — Gradient w.r.t. b: `dj_db = sum(err) / m`

$$\frac{\partial J}{\partial b} = \frac{0.0 + (-0.4) + (-0.8)}{3} = -0.4$$

### Shape Summary

| Quantity | Shape | Role |
|---|---|---|
| $\mathbf{err}$ | $(m,)$ | one error per example |
| $X^T$ | $(n,m)$ | features × examples |
| $X^T\mathbf{err}$ | $(n,)$ | one gradient value per feature; the $m$ dimension is summed away by the matmul, so no explicit loop is needed |

In [23]:
def compute_gradient(X, y, w, b):
    m = X.shape[0]                    # m = number of examples

    err = np.dot(X, w) + b - y        # computes ALL m errors at once — see below
    dj_dw = np.dot(X.T, err) / m      # sums err against each feature column — see below
    dj_db = np.sum(err) / m           # np.sum literally adds up every element of err

    return dj_db, dj_dw


<a name="toc_15456_5.2"></a>
## 5.2 Gradient Descent With Multiple Variables
*(5)*:
$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$


The routine below implements equation (5) above.

#### `gradient_descent` — Explained

Math being implemented (simultaneous update, repeated `num_iters` times):

$$w_j \leftarrow w_j - \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \qquad b \leftarrow b - \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}$$

In [24]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters  
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,)) : Updated values of parameters 
      b (scalar)       : Updated value of parameter 
      """
    
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_db,dj_dw = gradient_function(X, y, w, b)   

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
        
    return w, b #return final w,b 

In [25]:
# initialize parameters
initial_w = np.zeros_like(w_init)
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 5.0e-7

w_final, b_final= gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")

b,w found by gradient descent: -0.00,[ 0.20396569  0.00374919 -0.0112487  -0.0658614 ] 
